# Import Dependency

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from collections import Counter
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load Data and Map Labels

In [2]:
# 1. Load the data using Pandas
train_df = pd.read_csv("data/laptop_train.csv")


# 2. Create a dictionary to map string labels to numbers
label_mapping = {
    'negative': 0,
    'neutral': 1,
    'positive': 2,
    'conflict': 3
}

# 3. Apply the mapping to our dataframes to create a new numerical label column
train_df['label_id'] = train_df['label'].map(label_mapping)

In [3]:
train_df.head()

,text,span,label,ordinal,label_id
0,I charge it at night and skip taking the cord ...,cord,neutral,0,1
1,I charge it at night and skip taking the cord ...,battery life,positive,0,2
2,The tech guy then said the service center does...,service center,negative,0,0
3,The tech guy then said the service center does...,"""sales"" team",negative,0,0
4,The tech guy then said the service center does...,tech guy,neutral,0,1


# Tokenization

In [4]:
def tokenize(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove special characters using regex (keeps only letters and numbers)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Split the text by spaces into a list of words
    return text.split()

# Building a Vocabulary

In [5]:
def build_vocab(sentences):
    # Count how many times each word appears
    word_counts = Counter()
    for sentence in sentences:
        tokens = tokenize(sentence)
        word_counts.update(tokens)

    # <PAD>: Used to make all sentences the same length. (Index 0)
    # <UNK>: Used for "Unknown" words that we haven't seen before. (Index 1)
    vocab = {"<PAD>": 0, "<UNK>": 1}

    # Give a unique number to words
    index = 2
    for word in word_counts:
        vocab[word] = index
        index += 1

    return vocab

In [6]:
# Now let's use the function to build the vocabulary from our training data
vocab = build_vocab(train_df['text'])
print(f"Total words in our vocabulary: {len(vocab)}")

Total words in our vocabulary: 3191


# Encoding and Padding Sentences

In [7]:
def encode_sentence(sentence, vocab, max_length):
    # 1. Tokenize the sentence into words
    tokens = tokenize(sentence)

    # 2. Convert words to their vocabulary numbers
    encoded = []
    for word in tokens:
        if word in vocab:
            encoded.append(vocab[word])
        else:
            encoded.append(vocab["<UNK>"])

    # 3. Pad or truncate the sentence to make it exactly 'max_length'
    if len(encoded) < max_length:
        # Add <PAD> (which is 0) to the end until it reaches max_length
        padding = [vocab["<PAD>"]] * (max_length - len(encoded))
        encoded = padding + encoded

    return encoded

In [8]:
# Let's test it on a sample sentence!
sample_sentence = "This laptop is amazing!"
# We will use a max length of 10 words
print(encode_sentence(sample_sentence, vocab, max_length=10))

[0, 0, 0, 0, 0, 0, 167, 84, 38, 369]


 # Split Data using X and y

In [9]:
# 1. Define X (our text) and y (our numerical labels)
X = train_df['text']
y = train_df['label_id']

# 2. Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# 3. Combine them back into DataFrames for PyTorch
train_split_df = pd.DataFrame({'text': X_train, 'label_id': y_train})
test_split_df = pd.DataFrame({'text': X_test, 'label_id': y_test})

# Creating the PyTorch Dataset & DataLoader

In [11]:
# 1. Find the length of the longest sentence in the training data
max_length_in_data = 0

for sentence in train_df['text']:

    tokens = tokenize(sentence)

    if len(tokens) > max_length_in_data:
        max_length_in_data = len(tokens)

print(f"The longest sentence in training data has {max_length_in_data} words.")

MAX_LENGTH = max_length_in_data

The longest sentence in training data has 78 words.


In [12]:
# Create a Custom Dataset class that inherits from PyTorch's Dataset
class SentimentDataset(Dataset):
    def __init__(self, dataframe, vocab, max_length):
        self.dataframe = dataframe
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        # Returns the total number of rows in our dataframe
        return len(self.dataframe)

    def __getitem__(self, index):
        # 1. Grab the specific row from our dataframe
        row = self.dataframe.iloc[index]
        text = row['text']
        label = row['label_id']

        # 2. Convert text to a padded list of numbers using our function from Cell 5
        encoded_text = encode_sentence(text, self.vocab, self.max_length)

        # 3. Convert both the text array and label into PyTorch Tensors
        text_tensor = torch.tensor(encoded_text, dtype=torch.long)
        label_tensor = torch.tensor(label, dtype=torch.long)

        return text_tensor, label_tensor

In [13]:
# 2. Create the Datasets using our calculated MAX_LENGTH
train_dataset = SentimentDataset(train_split_df, vocab, MAX_LENGTH)
test_dataset = SentimentDataset(test_split_df, vocab, MAX_LENGTH)

In [14]:
# 3. Create the DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [15]:
print(f"Number of training batches: {len(train_loader)}")

Number of training batches: 59


# Defining the LSTM Architecture

In [16]:
class SentimentLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):

        super(SentimentLSTM, self).__init__()

        # 1. Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # 2. LSTM Layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)

        # 3. Linear / Output Layer
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):

        # Step 1: Pass the raw numbers through the embedding layer
        embedded = self.embedding(text)

        # Step 2: Pass the embeddings into the LSTM
        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Step 3: Pass that final memory into the Linear layer to get our predictions
        predictions = self.fc(hidden[-1])

        return predictions

# Initializing the Model

In [17]:
# Hyperparameters (You can experiment with changing these later!)
vocab_size = len(vocab)
embedding_dim = 100    # The size of the word meaning vectors
hidden_dim = 256       # The size of the LSTM's memory
output_dim = 4         # We have 4 classes: negative (0), neutral (1), positive (2), conflict (3)
num_layer = 2         # How many LSTMs to stack on top of each other (2 is standard)

# Create the model
model = SentimentLSTM(vocab_size, embedding_dim, hidden_dim, output_dim, num_layer)

In [18]:
# Move the model to the GPU if you have one, otherwise use the CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model is running on: {device}")
print(model)

Model is running on: cpu
SentimentLSTM(
  (embedding): Embedding(3191, 100, padding_idx=0)
  (lstm): LSTM(100, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=4, bias=True)
)


# Setup Optimizer and Loss Function

In [19]:
learning_rate = 0.001

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# Move the loss function to our device (GPU/CPU)
criterion = criterion.to(device)

# Training Loop

In [20]:
epochs = 20

print("Starting Training...")

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for batch_texts, batch_labels in train_loader:

        batch_texts = batch_texts.to(device)
        batch_labels = batch_labels.to(device)

        # Step 1: Clear the old gradients
        optimizer.zero_grad()

        # Step 2: Forward Pass (Predict)
        predictions = model(batch_texts)

        # Step 3: Calculate the Loss
        loss = criterion(predictions, batch_labels)

        # Step 4: Backward Pass (Calculate adjustments)
        loss.backward()

        # Step 5: Optimizer Step (Apply adjustments)
        optimizer.step()

        total_loss += loss.item()

        # predictions contains 4 numbers for each sentence.
        _, predicted_classes = torch.max(predictions, dim=1)

        # Count how many predictions matched the real labels
        correct_predictions += (predicted_classes == batch_labels).sum().item()
        total_samples += batch_labels.size(0)

    # Calculate the average loss and accuracy for this epoch
    epoch_loss = total_loss / len(train_loader)
    epoch_acc = (correct_predictions / total_samples) * 100

    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Training Accuracy: {epoch_acc:.2f}%")

Starting Training...
Epoch 1/20 | Loss: 1.1046 | Training Accuracy: 50.27%
Epoch 2/20 | Loss: 0.9426 | Training Accuracy: 59.97%
Epoch 3/20 | Loss: 0.8034 | Training Accuracy: 67.82%
Epoch 4/20 | Loss: 0.6814 | Training Accuracy: 73.33%
Epoch 5/20 | Loss: 0.5683 | Training Accuracy: 78.42%
Epoch 6/20 | Loss: 0.4666 | Training Accuracy: 82.03%
Epoch 7/20 | Loss: 0.3933 | Training Accuracy: 84.94%
Epoch 8/20 | Loss: 0.3527 | Training Accuracy: 86.21%
Epoch 9/20 | Loss: 0.3169 | Training Accuracy: 87.86%
Epoch 10/20 | Loss: 0.2890 | Training Accuracy: 87.54%
Epoch 11/20 | Loss: 0.2504 | Training Accuracy: 89.13%
Epoch 12/20 | Loss: 0.2290 | Training Accuracy: 89.66%
Epoch 13/20 | Loss: 0.2307 | Training Accuracy: 89.34%
Epoch 14/20 | Loss: 0.2138 | Training Accuracy: 89.93%
Epoch 15/20 | Loss: 0.2072 | Training Accuracy: 89.71%
Epoch 16/20 | Loss: 0.1999 | Training Accuracy: 89.50%
Epoch 17/20 | Loss: 0.1887 | Training Accuracy: 89.98%
Epoch 18/20 | Loss: 0.1842 | Training Accuracy: 89.98

# Evaluation

In [21]:
print("Evaluating Model on Test Data...")

model.eval()

correct_predictions = 0
total_samples = 0

all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch_texts, batch_labels in test_loader:

        batch_texts = batch_texts.to(device)
        batch_labels = batch_labels.to(device)

        predictions = model(batch_texts)

        _, predicted_classes = torch.max(predictions, dim=1)

        correct_predictions += (predicted_classes == batch_labels).sum().item()
        total_samples += batch_labels.size(0)

        # Save predictions and actual labels
        all_predictions.extend(predicted_classes.cpu().numpy())
        all_true_labels.extend(batch_labels.cpu().numpy())
        # .cpu().numpy() moves the data off the GPU and converts it to standard Python lists

# Calculate the final test accuracy
test_accuracy = (correct_predictions / total_samples) * 100
print(f"Test Accuracy: {test_accuracy:.2f}%\n")

# Print the Detailed Report Card
target_names_dict = {0: 'negative', 1: 'neutral', 2: 'positive', 3: 'conflict'}
ordered_names = [target_names_dict[i] for i in range(4)]

print("Detailed Classification Report:")
print(classification_report(all_true_labels, all_predictions, target_names=ordered_names))

Evaluating Model on Test Data...
Test Accuracy: 66.74%

Detailed Classification Report:
              precision    recall  f1-score   support

    negative       0.67      0.70      0.68       179
     neutral       0.51      0.51      0.51        92
    positive       0.77      0.75      0.76       189
    conflict       0.12      0.08      0.10        12

    accuracy                           0.67       472
   macro avg       0.52      0.51      0.51       472
weighted avg       0.66      0.67      0.67       472



In [22]:
def predict_sentiment(sentence):
    # Put model in evaluation mode
    model.eval()

    # 1. Encode and pad the custom sentence (using our dynamic MAX_LENGTH)
    # We use our pre-padding trick here!
    encoded = encode_sentence(sentence, vocab, MAX_LENGTH)

    # 2. Convert to PyTorch tensor and add a fake "batch" dimension of size 1
    # The model expects shape [batch_size, length], so we use unsqueeze(0) to make it [1, MAX_LENGTH]
    tensor = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)

    # 3. Predict!
    with torch.no_grad():
        prediction = model(tensor)

        # Get the highest predicted class
        _, predicted_class = torch.max(prediction, dim=1)

    # Map it back to the string name
    target_names_dict = {0: 'negative', 1: 'neutral', 2: 'positive', 3: 'conflict'}
    result = target_names_dict[predicted_class.item()]

    print(f'Sentence: "{sentence}"')
    print(f'Predicted Sentiment: {result.upper()}')

# Try it out!
predict_sentiment("This laptop is incredibly fast but the battery life is terrible.")
predict_sentiment("I absolutely love this new machine.")
predict_sentiment("It crashes every time I try to open a file.")
predict_sentiment("This laptop runs so hot it burned my lap, but the screen is beautiful.")
predict_sentiment("I really don't know why everyone hates this computer, it works totally fine for me.")

Sentence: "This laptop is incredibly fast but the battery life is terrible."
Predicted Sentiment: POSITIVE
Sentence: "I absolutely love this new machine."
Predicted Sentiment: POSITIVE
Sentence: "It crashes every time I try to open a file."
Predicted Sentiment: POSITIVE
Sentence: "This laptop runs so hot it burned my lap, but the screen is beautiful."
Predicted Sentiment: POSITIVE
Sentence: "I really don't know why everyone hates this computer, it works totally fine for me."
Predicted Sentiment: NEUTRAL


# Save Your Model

In [23]:
# Save the model's weights
model_save_path = 'laptop_sentiment_lstm.pth'
torch.save(model.state_dict(), model_save_path)

print(f"Model saved successfully to {model_save_path}!")

Model saved successfully to laptop_sentiment_lstm.pth!
